# Homework 3: Vaccine Warehouse Binary

###  Bus 36109 "Advanced Decision Modeling with Python", Don Eisenstein
Don Eisenstein &copy; Copyright 2024, University of Chicago 

## Instructions

A vaccine is being produced at different manufacturing Plants to be distributed across the country to various Hospitals. Vaccines must travel through Warehouses along the way to a Hospital. 

Vaccines are transported in units of cases.  The capacity of a plant is in units of cases per week.  Each Warehouse must process and ship all the cases it receives each week.  Each Warehouse has a capacity of cases it can process each week and a processing cost to process each case.  Each Warehouse also has a fixed weekly operating cost that is incurred if it processes any cases of vaccine.  If a Warehouse processes no vaccine then its fixed cost is not incurred (that is, we can close a warehouse down and not incur its fixed cost). 

Each Hospital has a demand for vaccine cases each week.  The supply of vaccine to a hospital must not exceed its demand. 

The transportation cost is \$1 per vaccine case traveling one unit of distance between plants and warehouses and between warehouses and hospitals.

All facilites, distances and costs are described in an Airtable base.

Each manufacturing Plant must ship as many doses as possible in an attempt to meet, but not exceed, Hospital demand.  A manufacturing Plant can ship vaccines to multiple Warehouses, a Warehouse can ship vaccines to multiple Hospitals, and a Hospital can receive vaccines from multiple Warehouses.   

Your code should be robust, that is, it should NOT make any assumptions about the total plant and warehouse capacities, or total hospital demands. 

Your model should find the optimal vaccine flows from Plants to Warehouses and Warehouses to Hospitals to minimize the weekly transportation, processing and operating costs, while sending as much vaccine as is feasibly possible.  And in doing so, determining which Warehouses are Open and which Closed.

**IMPORTANT:** Model your flow of vaccine cases as a continuous variable.  That is, you can ship fractions of cases between facilities.

**NOTE** No single variable carrying flow should include the ENTIRE path from a Plant through a Warehouse and into a Hospital.   That is, you must have a set of simple flow variables from Plants to Warehouses, and another set of simple flow variables from Warehouses to Hospitals.

Follow the notebook to walk you through the solution in parts.  Insert your answer to each part into the notebook below the question for each part.   Turn in your completed notebook with all output visible.

# Your Solution

Insert your answer to each part into this notebook

In [1]:
# === SETUP ===
import pulp
import os
from pyairtable import Api
from pprint import pprint

# Portable solver setup, to allow work locally (ARM64 architecture) as well as in JupyterHub
from pulp import COIN_CMD
if os.path.exists('/opt/homebrew/opt/cbc/bin/cbc'):
    solver = COIN_CMD(path='/opt/homebrew/opt/cbc/bin/cbc', msg=0)
else:
    solver = pulp.PULP_CBC_CMD(msg=0)

In [ ]:
AIRTABLE_API_TOKEN = #token
AIRTABLE_BASE_ID = 'appWITdHvwKWupWZR' # The ID for the HW3 data table -- don't change this
api = Api(AIRTABLE_API_TOKEN)

<!-- BEGIN QUESTION -->

**1. In broad terms, what are the variables, objective and constraints of this problem? You don't need to list the entire formulation. Just describe the structure/characteristics of your model.**

### Sets:
- Plants ($i = 1, 2, 3, \dots, m$)
- Warehouses ($j = 1, 2, 3, \dots, n$)
- Hospitals ($k = 1, 2, 3, \dots, p$)

### Data:
- $\text{capacity}_i$: capacity of plant $i$
- $\text{capacity}_j$: capacity of warehouse $j$
- $\text{demand}_k$: demand of hospital $k$
- $\text{distance}_{ij}$: distance from plant $i$ to warehouse $j$
- $\text{distance}_{jk}$: distance from warehouse $j$ to hospital $k$
- $\text{processing cost}_j$: per-case processing cost at warehouse $j$
- $\text{fixed cost}_j$: fixed weekly operating cost of warehouse $j$

### Decision Variables:
- $\text{cases}_{ij} \geq 0$: cases sent from plant $i$ to warehouse $j$
- $\text{cases}_{jk} \geq 0$: cases sent from warehouse $j$ to hospital $k$
- $\text{bool open}_j \in \{0, 1\}$: 1 if warehouse $j$ is open, 0 otherwise

### Objective:

#### Maximize total vaccine delivered minus total cost (using a large factor `Factor` to prioritize delivery):

$$
\max \left( \text{Factor} \cdot \sum_{j=1}^{n} \sum_{k=1}^{p} \text{cases}_{jk} - C_{\text{total}} \right)
$$

where total cost is:

$$
C_{\text{transport}} = \sum_{i=1}^{m} \sum_{j=1}^{n} \text{distance}_{ij} \cdot \text{cases}_{ij} + \sum_{j=1}^{n} \sum_{k=1}^{p} \text{distance}_{jk} \cdot \text{cases}_{jk}
$$

$$
C_{\text{processing}} = \sum_{j=1}^{n} \text{processing cost}_j \cdot \sum_{k=1}^{p} \text{cases}_{jk}
$$

$$
C_{\text{fixed}} = \sum_{j=1}^{n} \text{fixed cost}_j \cdot \text{bool open}_j
$$

$$
C_{\text{total}} = C_{\text{transport}} + C_{\text{processing}} + C_{\text{fixed}}
$$

### Constraints:

#### 1. For each plant $(i = 1, ..., m)$, the sum of all cases traveling to warehouses $(j = 1, ..., n)$ from it must not exceed its capacity.
$$
\sum_{j=1}^{n} \text{cases}_{ij} \leq \text{capacity}_i \quad \forall\ \text{Plant: } i = 1, 2, 3, \dots, m
$$

#### 2. Each warehouse $(j = 1, ..., n)$, *if it's open*, processes at most its capacity, and none otherwise.
$$
\sum_{i=1}^{m} \text{cases}_{ij} \leq \text{capacity}_j \cdot \text{bool open}_j \quad \forall\ \text{Warehouse: } j = 1, 2, 3, \dots, n
$$

#### 3. For each hospital $(k = 1, ..., p)$, the sum of all cases traveling from warehouses $(j = 1, ..., n)$ must not exceed its demand.
$$
\sum_{j=1}^{n} \text{cases}_{jk} \leq \text{demand}_k \quad \forall\ \text{Hospital: } k = 1, 2, 3, \dots, p
$$

#### 4. For each warehouse $(j = 1, ..., n)$, the sum of all cases traveling to it from plants $(i = 1, ..., m)$ from it must equal the sum of all cases traveling to all hospitals $(k = 1, ..., p)$
$$
\sum_{i=1}^{m} \text{cases}_{ij} = \sum_{k=1}^{p} \text{cases}_{jk} \quad \forall\ \text{Warehouse: } j = 1, 2, 3, \dots, n
$$

#### 5. Non-negativity and binary value constraints
$$
cases_{ij} \geq 0, \quad cases_{jk} \geq 0, \quad \text{bool open}_j \in \{0,1\}
$$

**2. Click on this link to access the data on AirTable: [AirTable Data](https://airtable.com/invite/l?inviteId=invCF2IEwn3KB68tR&inviteToken=8feebbc273e675e663a026e7a321221cedbdd4d893229610e2df40a2488647ef&utm_medium=email&utm_source=product_team&utm_content=transactional-alerts)** 

You may have to add access to this airtable base onto your token if you did not add permission to ALL bases to start.

<!-- END QUESTION -->

**3. Read in the `plants` table from AirTable. Store in an appropriate structure in Python.  Print out your Python structure. Verify that the data looks as expected.**

In [3]:
plants_table = api.table(AIRTABLE_BASE_ID, 'plants')

plants = []
for plant in plants_table.all():
    plants.append(
        {'name' : plant['fields']['name'],
         'capacity' : plant['fields']['capacity']})
pprint(plants)

[{'capacity': 100, 'name': 'plant_6'},
 {'capacity': 100, 'name': 'plant_5'},
 {'capacity': 60, 'name': 'plant_2'},
 {'capacity': 150, 'name': 'plant_3'},
 {'capacity': 200, 'name': 'plant_1'},
 {'capacity': 40, 'name': 'plant_4'}]


**4.  Now read in, print, and verify the hospitals table.** 

In [4]:
hospitals_table = api.table(AIRTABLE_BASE_ID, 'hospitals')

hospitals = []
for hospital in hospitals_table.all():
    hospitals.append(
        {'name' : hospital['fields']['name'],
         'demand' : hospital['fields']['demand']})
pprint(hospitals)

[{'demand': 40, 'name': 'hospital_2'},
 {'demand': 70, 'name': 'hospital_10'},
 {'demand': 90, 'name': 'hospital_6'},
 {'demand': 50, 'name': 'hospital_1'},
 {'demand': 120, 'name': 'hospital_4'},
 {'demand': 100, 'name': 'hospital_7'},
 {'demand': 50, 'name': 'hospital_5'},
 {'demand': 90, 'name': 'hospital_8'},
 {'demand': 50, 'name': 'hospital_9'},
 {'demand': 90, 'name': 'hospital_3'}]


**5.  Now read in, print, and verify the warehouse table.** 

In [5]:
warehouse_table = api.table(AIRTABLE_BASE_ID, 'warehouses')

warehouses = []
for warehouse in warehouse_table.all():
    warehouses.append(
        {'name' : warehouse['fields']['name'],
         'capacity' : warehouse['fields']['capacity'],
         'processing_cost' : warehouse['fields']['processing_cost'],
         'fixed_cost' : warehouse['fields']['fixed_cost']})
pprint(warehouses)

[{'capacity': 100,
  'fixed_cost': 550,
  'name': 'warehouse_8',
  'processing_cost': 10},
 {'capacity': 80,
  'fixed_cost': 1200,
  'name': 'warehouse_5',
  'processing_cost': 25},
 {'capacity': 100,
  'fixed_cost': 310,
  'name': 'warehouse_4',
  'processing_cost': 30},
 {'capacity': 290,
  'fixed_cost': 1590,
  'name': 'warehouse_6',
  'processing_cost': 10},
 {'capacity': 80,
  'fixed_cost': 290,
  'name': 'warehouse_3',
  'processing_cost': 30},
 {'capacity': 50,
  'fixed_cost': 410,
  'name': 'warehouse_7',
  'processing_cost': 25},
 {'capacity': 200,
  'fixed_cost': 750,
  'name': 'warehouse_2',
  'processing_cost': 25},
 {'capacity': 200,
  'fixed_cost': 1000,
  'name': 'warehouse_1',
  'processing_cost': 20}]


**6.  Now read in, print, and verify the distances table.** 

In [6]:
distance_table = api.table(AIRTABLE_BASE_ID, 'distances')

distances = []
for distance in distance_table.all():
    distances.append(
        {'start' : distance['fields']['start'],
         'end' : distance['fields']['end'],
         'distance' : distance['fields']['distance']})
pprint(distances)

[{'distance': 284, 'end': 'hospital_3', 'start': 'warehouse_5'},
 {'distance': 269, 'end': 'hospital_9', 'start': 'warehouse_3'},
 {'distance': 106, 'end': 'warehouse_3', 'start': 'plant_2'},
 {'distance': 70, 'end': 'warehouse_6', 'start': 'plant_3'},
 {'distance': 213, 'end': 'hospital_5', 'start': 'warehouse_5'},
 {'distance': 228, 'end': 'hospital_5', 'start': 'warehouse_7'},
 {'distance': 364, 'end': 'hospital_1', 'start': 'warehouse_4'},
 {'distance': 209, 'end': 'hospital_9', 'start': 'warehouse_5'},
 {'distance': 125, 'end': 'hospital_4', 'start': 'warehouse_6'},
 {'distance': 315, 'end': 'warehouse_6', 'start': 'plant_6'},
 {'distance': 177, 'end': 'warehouse_6', 'start': 'plant_1'},
 {'distance': 191, 'end': 'hospital_3', 'start': 'warehouse_4'},
 {'distance': 110, 'end': 'hospital_6', 'start': 'warehouse_7'},
 {'distance': 121, 'end': 'hospital_4', 'start': 'warehouse_8'},
 {'distance': 10, 'end': 'hospital_2', 'start': 'warehouse_3'},
 {'distance': 229, 'end': 'hospital_10'

**7. Create a PuLP LpProblem object and store it in the variable `model`.** 

In [7]:
model = pulp.LpProblem('Vaccine_Binary', pulp.LpMaximize)

**8. Create and store your PuLP decision variables. Print out your Python structures that hold them.**


In [8]:
# Start with an empty dictionary of cases shipped
shipments = {}

# Iterate over plant-warehouse pairs (i-j) and create LpVariables for each
# Key: tuple(plant_name, hospital_name)
# Value: LpVariable representing the number of cases shipped on that route
for plant in plants:
    for warehouse in warehouses:
        # Define pair
        pair = (plant['name'],warehouse['name'])
        # Create PuLP variable for the pair and put in the dictionary
        var_name = f"Cases_{plant['name']}_{warehouse['name']}"
        shipments[pair] = pulp.LpVariable(var_name, lowBound = 0, cat='Continuous')

# Iterate over warehouse-hospital pairs (j-k)
for warehouse in warehouses:
    for hospital in hospitals:
        # Define pair
        pair = (warehouse['name'],hospital['name'])
        # Create PuLP variable for the pair and put in the dictionary
        var_name = f"Cases_{warehouse['name']}_{hospital['name']}"
        shipments[pair] = pulp.LpVariable(var_name, lowBound = 0, cat='Continuous')

# Build a dictionary of warehouse open/closed status
bool_open = {}
for warehouse in warehouses:
    var_name = f"{warehouse['name']}_open"
    bool_open[warehouse['name']] = pulp.LpVariable(var_name, cat='Binary')

**9. Add your objective function to your `model`.  Print your model to verify**

In [30]:
# Primary objective: send as many vaccines as possible
total_shipment = 0

for warehouse in warehouses:
    for hospital in hospitals:
        pair = (warehouse['name'], hospital['name'])
        total_shipment += shipments[pair]

print(f"{total_shipment}")

Cases_warehouse_1_hospital_1 + Cases_warehouse_1_hospital_10 + Cases_warehouse_1_hospital_2 + Cases_warehouse_1_hospital_3 + Cases_warehouse_1_hospital_4 + Cases_warehouse_1_hospital_5 + Cases_warehouse_1_hospital_6 + Cases_warehouse_1_hospital_7 + Cases_warehouse_1_hospital_8 + Cases_warehouse_1_hospital_9 + Cases_warehouse_2_hospital_1 + Cases_warehouse_2_hospital_10 + Cases_warehouse_2_hospital_2 + Cases_warehouse_2_hospital_3 + Cases_warehouse_2_hospital_4 + Cases_warehouse_2_hospital_5 + Cases_warehouse_2_hospital_6 + Cases_warehouse_2_hospital_7 + Cases_warehouse_2_hospital_8 + Cases_warehouse_2_hospital_9 + Cases_warehouse_3_hospital_1 + Cases_warehouse_3_hospital_10 + Cases_warehouse_3_hospital_2 + Cases_warehouse_3_hospital_3 + Cases_warehouse_3_hospital_4 + Cases_warehouse_3_hospital_5 + Cases_warehouse_3_hospital_6 + Cases_warehouse_3_hospital_7 + Cases_warehouse_3_hospital_8 + Cases_warehouse_3_hospital_9 + Cases_warehouse_4_hospital_1 + Cases_warehouse_4_hospital_10 + Case

In [29]:
# Secondary objective: In achieving the above, minimize cost
cost_pw = 0.0
cost_wh = 0.0
cost_p = 0.0
cost_f = 0.0

# Transportation cost between plants & warehouses
for plant in plants:
    for warehouse in warehouses:
        for distance in distances:
            pair = (plant['name'],warehouse['name'])
            if distance['start'] == plant['name'] and \
            distance['end'] == warehouse['name']:
                cost_pw += distance['distance'] * shipments[pair]

# Transportation cost between warehouses & hospitals
for warehouse in warehouses:
    for hospital in hospitals:
        for distance in distances:
            pair = (warehouse['name'],hospital['name'])
            if distance['start'] == warehouse['name'] and \
            distance['end'] == hospital['name']:
                cost_wh += distance['distance'] * shipments[pair]

# Warehouse variable processing cost
for warehouse in warehouses:
    for hospital in hospitals:
        pair = (warehouse['name'],hospital['name'])
        cost_p += warehouse['processing_cost'] * shipments[pair]

# Warehouse fixed cost
for warehouse in warehouses:
    cost_f += warehouse['fixed_cost'] * bool_open[warehouse['name']]

print(f"{cost_pw} \n")
print(f"{cost_wh} \n")
print(f"{cost_p} \n")
print(f"{cost_f}")

203*Cases_plant_1_warehouse_1 + 270*Cases_plant_1_warehouse_2 + 162*Cases_plant_1_warehouse_3 + 129*Cases_plant_1_warehouse_4 + 222*Cases_plant_1_warehouse_5 + 177*Cases_plant_1_warehouse_6 + 17*Cases_plant_1_warehouse_7 + 213*Cases_plant_1_warehouse_8 + 19*Cases_plant_2_warehouse_1 + 214*Cases_plant_2_warehouse_2 + 106*Cases_plant_2_warehouse_3 + 315*Cases_plant_2_warehouse_4 + 166*Cases_plant_2_warehouse_5 + 63*Cases_plant_2_warehouse_6 + 219*Cases_plant_2_warehouse_7 + 157*Cases_plant_2_warehouse_8 + 138*Cases_plant_3_warehouse_1 + 347*Cases_plant_3_warehouse_2 + 239*Cases_plant_3_warehouse_3 + 220*Cases_plant_3_warehouse_4 + 299*Cases_plant_3_warehouse_5 + 70*Cases_plant_3_warehouse_6 + 124*Cases_plant_3_warehouse_7 + 290*Cases_plant_3_warehouse_8 + 98*Cases_plant_4_warehouse_1 + 111*Cases_plant_4_warehouse_2 + 13*Cases_plant_4_warehouse_3 + 240*Cases_plant_4_warehouse_4 + 63*Cases_plant_4_warehouse_5 + 166*Cases_plant_4_warehouse_6 + 144*Cases_plant_4_warehouse_7 + 60*Cases_plant_

In [11]:
factor = 10**10 # To ensure primary objective gets priority
model += factor * total_shipment - (cost_pw + cost_wh + cost_p + cost_f)

**10. Add all of your constraints to your `model`.   Use python comments to document each type of constraint before you add them.**

In [12]:
# Constraint set 1: Each plant ships no more than its capacity
for plant in plants:
    total_shipped = 0
    for warehouse in warehouses:
        total_shipped += shipments[(plant['name'], warehouse['name'])]
    model += total_shipped <= plant['capacity'], f"Plant_Capacity_{plant['name']}"

# Constraint set 2: Each warehouse receives no more than its capacity, and only if it's open
for warehouse in warehouses:
    total_received = 0
    for plant in plants:
        total_received += shipments[(plant['name'], warehouse['name'])]
    model += total_received <= warehouse['capacity'] * bool_open[warehouse['name']], f"Warehouse_Capacity_{warehouse['name']}"

# Constraint set 3: Each hospital receives no more than its demand
for hospital in hospitals:
    total_received = 0
    for warehouse in warehouses:
        total_received += shipments[(warehouse['name'], hospital['name'])]
    model += total_received <= hospital['demand'], f"Hospital_Demand_{hospital['name']}"

# Constraint set 4: Flow conservation at each warehouse (in = out)
for warehouse in warehouses:
    inflow = 0
    for plant in plants:
        inflow += shipments[(plant['name'], warehouse['name'])]
    outflow = 0
    for hospital in hospitals:
        outflow += shipments[(warehouse['name'], hospital['name'])]
    model += inflow == outflow, f"Flow_Conservation_{warehouse['name']}"

# Constraint set 5: Non-negative integers handled by LpVariable definition (lowBound=0, cat='Continuous') # or Integer

<!-- BEGIN QUESTION -->

**11. Display your model with `print(model)`, check that all is OK**

In [13]:
print(model)

Vaccine_Binary:
MAXIMIZE
-203*Cases_plant_1_warehouse_1 + -270*Cases_plant_1_warehouse_2 + -162*Cases_plant_1_warehouse_3 + -129*Cases_plant_1_warehouse_4 + -222*Cases_plant_1_warehouse_5 + -177*Cases_plant_1_warehouse_6 + -17*Cases_plant_1_warehouse_7 + -213*Cases_plant_1_warehouse_8 + -19*Cases_plant_2_warehouse_1 + -214*Cases_plant_2_warehouse_2 + -106*Cases_plant_2_warehouse_3 + -315*Cases_plant_2_warehouse_4 + -166*Cases_plant_2_warehouse_5 + -63*Cases_plant_2_warehouse_6 + -219*Cases_plant_2_warehouse_7 + -157*Cases_plant_2_warehouse_8 + -138*Cases_plant_3_warehouse_1 + -347*Cases_plant_3_warehouse_2 + -239*Cases_plant_3_warehouse_3 + -220*Cases_plant_3_warehouse_4 + -299*Cases_plant_3_warehouse_5 + -70*Cases_plant_3_warehouse_6 + -124*Cases_plant_3_warehouse_7 + -290*Cases_plant_3_warehouse_8 + -98*Cases_plant_4_warehouse_1 + -111*Cases_plant_4_warehouse_2 + -13*Cases_plant_4_warehouse_3 + -240*Cases_plant_4_warehouse_4 + -63*Cases_plant_4_warehouse_5 + -166*Cases_plant_4_wareho

<!-- END QUESTION -->

**12. Solve your optimization model and print its status and the optimal objective function value.  The optimal objective function value is 117910.0**

In [21]:
model.solve(solver)
print(f"Status: {pulp.LpStatus[model.status]}")

Status: Optimal


In [24]:
total_cost = pulp.value(cost_pw + cost_wh + cost_p + cost_f)

print(f"Total cost = ${total_cost}")
print(f"Total shipment = {pulp.value(total_shipment)} cases")

Total cost = $117910.0
Total shipment = 650.0 cases


**13. Output the value of each of your variables at optimality.  Only print variables with non-zero value.**

In [15]:
for v in model.variables():
    print(f"{v.name} = {v.varValue}")

Cases_plant_1_warehouse_1 = 70.0
Cases_plant_1_warehouse_2 = 0.0
Cases_plant_1_warehouse_3 = 40.0
Cases_plant_1_warehouse_4 = 0.0
Cases_plant_1_warehouse_5 = 0.0
Cases_plant_1_warehouse_6 = 40.0
Cases_plant_1_warehouse_7 = 50.0
Cases_plant_1_warehouse_8 = 0.0
Cases_plant_2_warehouse_1 = 60.0
Cases_plant_2_warehouse_2 = 0.0
Cases_plant_2_warehouse_3 = 0.0
Cases_plant_2_warehouse_4 = 0.0
Cases_plant_2_warehouse_5 = 0.0
Cases_plant_2_warehouse_6 = 0.0
Cases_plant_2_warehouse_7 = 0.0
Cases_plant_2_warehouse_8 = 0.0
Cases_plant_3_warehouse_1 = 0.0
Cases_plant_3_warehouse_2 = 0.0
Cases_plant_3_warehouse_3 = 0.0
Cases_plant_3_warehouse_4 = 0.0
Cases_plant_3_warehouse_5 = 0.0
Cases_plant_3_warehouse_6 = 150.0
Cases_plant_3_warehouse_7 = 0.0
Cases_plant_3_warehouse_8 = 0.0
Cases_plant_4_warehouse_1 = 0.0
Cases_plant_4_warehouse_2 = 0.0
Cases_plant_4_warehouse_3 = 40.0
Cases_plant_4_warehouse_4 = 0.0
Cases_plant_4_warehouse_5 = 0.0
Cases_plant_4_warehouse_6 = 0.0
Cases_plant_4_warehouse_7 = 0.0


**14. Use Python to loop through each Hospital, display the total vaccine supplied to each hospital, its demand, shorfall, and percent of demand met.**

In [25]:
for hospital in hospitals:
    supply = 0
    for warehouse in warehouses:
        supply += shipments[(warehouse['name'], hospital['name'])].varValue
    demand = hospital['demand']
    shortfall = demand - supply
    percent_filled = (supply / demand) * 100
    print(f"{hospital['name']}: Demand={demand}, Supply={supply:.0f}, Shortfall={shortfall:.0f}, Filled={percent_filled:.1f}%")

hospital_2: Demand=40, Supply=40, Shortfall=0, Filled=100.0%
hospital_10: Demand=70, Supply=70, Shortfall=0, Filled=100.0%
hospital_6: Demand=90, Supply=30, Shortfall=60, Filled=33.3%
hospital_1: Demand=50, Supply=50, Shortfall=0, Filled=100.0%
hospital_4: Demand=120, Supply=120, Shortfall=0, Filled=100.0%
hospital_7: Demand=100, Supply=100, Shortfall=0, Filled=100.0%
hospital_5: Demand=50, Supply=50, Shortfall=0, Filled=100.0%
hospital_8: Demand=90, Supply=90, Shortfall=0, Filled=100.0%
hospital_9: Demand=50, Supply=10, Shortfall=40, Filled=20.0%
hospital_3: Demand=90, Supply=90, Shortfall=0, Filled=100.0%
